(custom-model)=

# Make Custom Models

``ELISA`` lets you define your own spectral components when the built-in
models or XSPEC wrappers are not sufficient. In practice there are two main
paths:

- use `AnaIntAdditive` / `NumIntAdditive` (or their multiplicative
  counterparts) when the model can be written directly with JAX operations;
- use `PyAnaInt` / `PyNumInt` when you need to wrap pure Python, NumPy, or
  other legacy numerical code.

This tutorial demonstrates both patterns with lightweight examples.

## 1. Import the Building Blocks

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from elisa import ParamConfig, PyAnaInt
from elisa.models import NumIntAdditive, PhAbs, PowerLaw

Every custom component needs three ingredients:

1. a `_config` tuple made of `ParamConfig` objects;
2. a `type` class attribute, usually `'add'` or `'mul'`;
3. a static model expression implemented as an `integral` or `continuum` method.

When the expression is already JAX-compatible, prefer the native JAX route,
because it integrates better with compilation and automatic differentiation.

## 2. A JAX-Native Additive Component

In [ ]:
class CutoffPowerLaw(NumIntAdditive):
    _config = (
        ParamConfig('alpha', r'\alpha', '', 1.5, -3.0, 5.0),
        ParamConfig('Ec', r'E_\mathrm{c}', 'keV', 200.0, 1.0, 1e4, True),
        ParamConfig('K', 'K', 'ph cm^-2 s^-1 keV^-1', 1.0, 1e-10, 1e10, True),
    )
    type: str = 'add'

    @staticmethod
    def continuum(egrid, params):
        alpha = params['alpha']
        ec = params['Ec']
        K = params['K']
        return K * jnp.power(egrid, -alpha) * jnp.exp(-egrid / ec)

In [ ]:
model = CutoffPowerLaw()
model

Once defined, the component behaves like any built-in additive model.

In [ ]:
compiled = model.compile()
egrid = np.geomspace(1.0, 1e3, 401)
emid = np.sqrt(egrid[:-1] * egrid[1:])

ne_default = compiled.ne(egrid)
ne_harder = compiled.ne(
    egrid,
    params={
        'CutoffPowerLaw.alpha': 1.0,
        'CutoffPowerLaw.Ec': 400.0,
    },
)

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(emid, ne_default, label='default')
ax.loglog(emid, ne_harder, label=r'$\\alpha=1.0, E_c=400\\,\\mathrm{keV}$')
ax.set_xlabel('Energy [keV]')
ax.set_ylabel(r'$N(E)$ [ph cm$^{-2}$ s$^{-1}$ keV$^{-1}$]')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

Custom components also compose with built-in ones in the usual way.

In [ ]:
absorbed_model = PhAbs() * CutoffPowerLaw()
absorbed_model

## 3. Wrapping Pure Python Code

If your model already exists as a NumPy or pure Python function, use
`PyAnaInt` or `PyNumInt`. These classes call your function through a JAX
callback and approximate derivatives numerically. That makes them convenient,
but generally slower than a native JAX implementation.

In [ ]:
class PythonPowerLaw(PyAnaInt):
    _config = (
        ParamConfig('alpha', r'\alpha', '', 1.01, -5.0, 5.0),
        ParamConfig('K', 'K', 'ph cm^-2 s^-1 keV^-1', 1.0, 1e-10, 1e10, True),
    )
    type: str = 'add'

    @staticmethod
    def integral(egrid, params):
        alpha = params['alpha']
        K = params['K']
        one_minus_alpha = 1.0 - alpha
        f = egrid**one_minus_alpha / one_minus_alpha
        return K * (f[1:] - f[:-1])

The example below compares the custom Python-backed model with the built-in
`PowerLaw` implementation.

In [ ]:
builtin = PowerLaw().compile()
python_model = PythonPowerLaw().compile()

params_builtin = {'PowerLaw.alpha': 1.7, 'PowerLaw.K': 2.0}
params_python = {'PythonPowerLaw.alpha': 1.7, 'PythonPowerLaw.K': 2.0}

np.allclose(
    builtin.eval(egrid, params_builtin),
    python_model.eval(egrid, params_python),
)

## 4. Practical Advice

- Use native JAX components whenever the model can be expressed with `jax.numpy`.
- Use `PyAnaInt` / `PyNumInt` to wrap legacy code or algorithms that are not
  easy to rewrite in JAX.
- Keep parameter defaults and bounds physically meaningful; this matters for
  both optimization and sampling.
- Test the compiled model on a representative energy grid before plugging it
  into a full fit.

After your custom component behaves correctly, it can be used anywhere a
built-in component is used: model composition, maximum-likelihood fitting,
Bayesian inference, simulations, and plotting.